In [1]:
import pandas as pd
from sqlalchemy import create_engine



### Load env variables

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
DATABASE_URL = os.environ['POSTGRES']

### Create POSTGRESQL connection

In [3]:

engine = create_engine(DATABASE_URL)

## Get the Raw Data from the properties table

In [4]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
"""

tables = pd.read_sql(query, engine)

tables

,table_name
0,train_data
1,val_data
2,test_data
3,properties
4,clean_properties


In [5]:
df = pd.read_sql("SELECT * FROM clean_properties", engine)

df_clean = df.copy()

# Cleaning Phase

## Area

In [ ]:
# Just a heads-up: when we’re cleaning the data, we won’t clean Area because it's already clean. However, there might be some work involved with it, and it really depends on the model we’re using. 

df_clean['area_log__area'].describe()

## Price

In [ ]:
# Just a heads-up: when we’re cleaning the data, we won’t clean price because it's already clean. However, there might be some work involved with it, and it really depends on the model we’re using. 

df_clean['price']

## Price per sqm 

In [ ]:
# create price_per_sqm from price and area

df_clean['price_per_sqm'] = df_clean['price'] / df_clean['area'].replace(0, pd.NA)

df_clean['price_per_sqm'].describe()
df_clean['price_per_sqm']

## Baths

In [ ]:
# convert to numeric
df_clean['baths'] = pd.to_numeric(
    df_clean['baths'],
    errors='coerce'
)

# impute using median inside each group
df_clean['baths'] = df_clean.groupby(
    ['property_type', 'area']
)['baths'].transform(
    lambda x: x.fillna(x.median())
)

# if any NaNs still exist -> drop them
df_clean = df_clean.dropna(subset=['baths'])
df_clean['baths']

## Studio

In [ ]:

# Detect studio apartments

df_clean['is_studio'] = (
    df_clean['beds']
    .astype(str)
    .str.contains('Studio', case=False, na=False)
)


# Convert Studio -> 0 
df_clean['beds'] = (
    df_clean['beds']
    .replace('Studio', 0)
)

df_clean['is_studio']


## Beds

In [ ]:
# Convert beds to numeric
df_clean['beds'] = pd.to_numeric(
    df_clean['beds'],
    errors='coerce'
)

# Group imputation
df_clean['beds'] = df_clean.groupby('property_type')['beds'].transform(
    lambda x: x.fillna(x.median())
)

df_clean['beds'] = df_clean['beds'].astype(int)

df_clean['beds']

## Link 
### For now, we’re keeping the LINK up, but we’ll probably take it down if we need to.

In [ ]:
df_clean['link']

## Source 
### For now, we’re keeping the SOURCE up, but we’ll probably take it down if we need to.

## Furnished

In [7]:
df_clean['furnishing'].value_counts()

furnishing
Unfurnished      8449
Not Specified    1552
Furnished         669
Name: count, dtype: int64

In [ ]:
# Let’s change all the “nulls” in Furnishing to “Not Specified.”

df_clean['furnishing'] = (
    df_clean['furnishing']
    .fillna('Not Specified')
)

df_clean['furnishing']

## Property type

In [6]:
# Normalize Property Type

# Remove extra spaces
# Standardize text casing
# Example:
# " apartment " -> "Apartment"

df_clean['property_type'] = (
    df_clean['property_type']
    .str.strip()
    .str.title()
)

print("Normalized property_type values:\n")
df_clean['property_type'].value_counts()

Normalized property_type values:



property_type
Apartment            8103
Villa                 829
Duplex                553
Penthouse             301
Townhouse             275
Stand Alone Villa     170
Twin House            138
Studio                118
Hotel Apartment        81
Chalet                 79
Other                  23
Name: count, dtype: int64

## Amenities

In [ ]:
# Fill nulls with Not Mentioned 

df_clean['amenities'] = (
    df_clean['amenities']
    .fillna('Not Mentioned')
)

# Parsing Amenities and Extract new columns (Amenities count & Amenities List)

def parse_amenities(text):

    if pd.isna(text) or text == 'Not Mentioned':
        return []

    return [
        item.strip()
        for item in str(text).split(',')
        if item.strip()
    ]

# Convert amenities string -> list
df_clean['amenities_list'] = (
    df_clean['amenities']
    .apply(parse_amenities)
)

# Count amenities
df_clean['amenity_count'] = (
    df_clean['amenities_list']
    .apply(len)
)

df_clean[['amenities_list','amenity_count','amenities']]

## Transaction type

In [ ]:
# Parse transaction type

df_clean['transaction_type'] = (
    df_clean['link']
    .str.contains('for-rent', case=False, na=False)
    .map({True: 'rent', False: 'sale'})
)

# Keeping only Sale Transactions and drop rent ones 
df_clean = df_clean[df_clean["transaction_type"] == "sale"]

df_clean['transaction_type']


## Location

In [25]:
# Just a heads-up: during the cleaning step, we’re not cleaning the location because it’s already clean. However, the amount of work involved might vary quite a bit depending on the model type.

df_clean['location'].value_counts().describe()

count    976.000000
mean      10.932377
std       37.708811
min        1.000000
25%        1.000000
50%        2.000000
75%        7.000000
max      858.000000
Name: count, dtype: float64

## City & District (Created out of Location)

## City & District with unknown as default (Created out of Location)

In [ ]:
# Extract district and city from location

KNOWN_CITIES = [
    'Cairo',
    'Giza',
    'Alexandria',
    'New Cairo',
    'New Capital City'
]

def parse_location(location):

    if pd.isna(location):
        return pd.NA, pd.NA

    parts = [p.strip() for p in location.split(',')]

    # First part usually compound / district
    district = parts[0]

    # Default
    city = 'Unknown'

    # Search for known cities
    for part in parts:
        if part in KNOWN_CITIES:
            city = part
            break

    return district, city


df_clean[['district', 'city']] = (
    df_clean['location']
    .apply(parse_location)
    .apply(pd.Series)
)

df_clean[['district', 'city']]

## Drop Columns 

### Just a heads-up: these columns aren’t really useful for building or improving machine learning models, so we’ve decided to remove them based on this reason .

## ID (drop)

In [ ]:
df_clean = df_clean.drop(columns=['id'])

## Checksum (drop)

In [ ]:
df_clean = df_clean.drop(columns=['checksum'])

## Title (drop)

In [ ]:
df_clean = df_clean.drop(columns=['title'])

## Reactivated date (drop)

In [ ]:
df_clean = df_clean.drop(columns=['reactivated_date'])

## Scraped at (drop)

In [ ]:
df_clean = df_clean.drop(columns=['scraped_at'])

## Transformed at (drop)

In [ ]:
df_clean = df_clean.drop(columns=['transformed_at'])

# Save the new Clean Data to a new table (clean properties) in the postgres db 

In [ ]:
df_clean.to_sql(
    name='clean_properties',
    con=engine,
    if_exists='replace',
    index=False
)

In [15]:

import pandas as pd
import re

# your location counts as a dataframe: columns ['location', 'count']
locs = pd.read_sql("SELECT location, COUNT(*) as cnt FROM clean_properties GROUP BY location", engine)

def canonicalize(loc):
    parts = [p.strip() for p in loc.split(',')]
    # drop trailing "Cairo"/"Giza"/governorate-only tokens
    while len(parts) > 2 and parts[-1] in ['Cairo', 'Giza', 'Red Sea', 'Matruh', 'Alexandria', 'Suez', 'Damietta', 'Ismailia']:
        parts = parts[:-1]
    # drop a duplicated city name if the second-to-last part repeats the city
    # e.g. "..., Shorouk City, Cairo" -> "..., Shorouk City"
    return ', '.join(parts)

locs['canonical'] = locs['location'].apply(canonicalize)


# build the map: original location -> canonical
loc_map = dict(zip(locs['location'], locs['canonical']))


GOVERNORATES = {'Cairo', 'Giza', 'Red Sea', 'Matruh', 'Alexandria', 'Suez', 'Damietta', 'Ismailia'}

def canonicalize(loc):
    parts = [p.strip() for p in loc.split(',')]
    # only drop the LAST part if it's a bare governorate name
    # AND there's something more specific before it (avoid nuking "Madinaty, Cairo" down to just "Madinaty")
    if len(parts) > 2 and parts[-1] in GOVERNORATES:
        parts = parts[:-1]
    return ', '.join(parts) 

# Step 1: apply your governorate-drop rule (as before)
locs['canonical'] = locs['location'].apply(canonicalize)

# Step 2: build a prefix -> full suffix lookup from the longer/more complete forms
# key = first 2 parts (compound, sub-area), value = the most common full canonical string sharing that prefix
def prefix_key(s):
    parts = [p.strip() for p in s.split(',')]
    return ', '.join(parts[:2])

locs['prefix'] = locs['canonical'].apply(prefix_key)

# for each prefix, find the longest canonical form (most specific) weighted by frequency
best_form = (locs.groupby('prefix')
             .apply(lambda g: g.loc[g['canonical'].str.len().idxmax(), 'canonical'])
             .to_dict())

# Step 3: remap anything short to its best known full form, if one exists
locss=locs['final'] = locs['prefix'].map(best_form).fillna(locs['canonical'])



# result = locs.groupby('final')['cnt'].sum().sort_values(ascending=False)
# print(result.head(30))
# print(f"\nTotal unique locations before: {locs['location'].nunique()}")
# print(f"Total unique locations after: {locs['final'].nunique()}")

# spot-check: locations whose prefix has multiple different 'canonical' values before merging
locs.groupby('prefix')['canonical'].nunique().sort_values(ascending=False).head(25)
locs[locs['prefix'].isin(['layan Compound, 5th Settlement', 'EL Patio ORO Compound, 5th Settlement'])][['location','canonical','cnt']]

/var/folders/1p/bsgt9ppn7sv8l7j2tn5j1gt40000gn/T/ipykernel_40368/858226524.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g['canonical'].str.len().idxmax(), 'canonical'])


,location,canonical,cnt
459,"layan Compound, 5th Settlement, New Cairo, Cairo","layan Compound, 5th Settlement, New Cairo",14
609,"EL Patio ORO Compound, 5th Settlement, New Cai...","EL Patio ORO Compound, 5th Settlement, New Cairo",46
733,"layan Compound, 5th Settlement","layan Compound, 5th Settlement",2
910,"EL Patio ORO Compound, 5th Settlement","EL Patio ORO Compound, 5th Settlement",165
